<a href="https://colab.research.google.com/github/fborrasumh/bioestadistica/blob/main/cuadernos/Tema_100_Pandas_valores_perdidos_y_atipicos_en_registros_clinicos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# <font color='steelblue'>Tema 100. Estructuras de datos con Pandas: tratamiento de valores perdidos y atípicos en registros clínicos</font>

**Bioestadística (1964) · Grado en Medicina · 2.º curso · Curso 2027-2028**

**Módulo 3. Procesado inicial de datos**

**Autor:** Fernando Borrás Rocher · Universidad Miguel Hernández de Elche · ORCID [0000-0002-5519-4573](https://orcid.org/0000-0002-5519-4573)

**Duración:** 50 minutos de clase, más el estudio de los apartados de ampliación

**Retos en la app:** [las tres misiones del Tema 100](https://fborrasumh.github.io/bioestadistica/#tema-100)

**Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es). Todos los pacientes y datos de este cuaderno son sintéticos.

*Antes de trabajar, guarda tu copia: **Archivo → Guardar una copia en Drive**.*

## <font color='steelblue'>Configuración del cuaderno</font>

Ejecuta esta celda al empezar. Instala el módulo de preguntas interactivas y prepara el cuaderno.

**Para dar la clase con el cuaderno:** usa **Entorno de ejecución → Ejecutar todo** antes de empezar, para que se dibujen los gráficos. Las celdas con borde azul son las diapositivas; el desarrollo de cada bloque está debajo y se puede plegar con la flecha de su título. El índice (☰, a la izquierda) sirve para saltar de un bloque a otro.

In [ ]:
#@title Ejecutar configuración
%pip install -q jupyterquiz==2.9.6.4
from jupyterquiz import display_quiz
print("Cuaderno configurado")

## <font color='steelblue'>Objetivos de aprendizaje</font>

Al terminar esta sesión serás capaz de:

1. Cargar un registro clínico en un **DataFrame** de Pandas y explorar su estructura.
2. Seleccionar columnas, filtrar filas y crear variables nuevas.
3. Detectar y corregir registros duplicados y categorías escritas de formas distintas.
4. Cuantificar los **valores perdidos** y decidir cómo tratarlos sin sesgar el análisis.
5. Distinguir un **error de registro** de un **valor atípico** real, con reglas de plausibilidad fisiológica y con el criterio del rango intercuartílico.
6. Documentar cada decisión de limpieza en un registro auditable.

## <font color='steelblue'>Plan de la sesión</font>

| Bloque | Minutos |
| --- | --- |
| 1. Caso clínico de partida | 5 |
| 2. Conceptos clave | 10 |
| 3. Demostración: limpieza de un registro clínico | 20 |
| 4. Práctica en la app Bioestadística | 10 |
| 5. Autoevaluación | 5 |

## <font color='steelblue'>1. Caso clínico de partida (5 min)</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">Un peso medio de 78 kg… con un paciente de 680 kg</p><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Una media se calcula siempre, aunque los datos estén mal: ningún error impide hacer la cuenta.</li><li>Antes de analizar, se limpia, y cada decisión queda documentada en un registro de limpieza.</li></ul></div>

In [ ]:
#@title Figura: Un peso medio de 78 kg… con un paciente de 680 kg
import pandas as pd
import matplotlib.pyplot as plt

datos = pd.read_csv("https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/datos/cohorte_cardiovascular.csv")
peso = datos["peso_kg"].dropna()
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.2))
ejes[0].hist(peso, bins=60, color="steelblue")
ejes[0].set_title("Peso registrado (kg)", fontweight="bold")
ejes[0].annotate("680 kg", xy=(680, 1), xytext=(520, 40), fontsize=13, color="firebrick", arrowprops=dict(arrowstyle="->", color="firebrick"))
ejes[0].annotate("7,4 kg", xy=(7.4, 1), xytext=(60, 60), fontsize=13, color="firebrick", arrowprops=dict(arrowstyle="->", color="firebrick"))
ejes[0].set_xlabel("kg"); ejes[0].set_ylabel("Pacientes")
problemas = {"Peso ausente": datos["peso_kg"].isna().sum(), "Sexo con varias grafías": datos["sexo"].nunique() - 2,
             "Pacientes duplicados": datos.duplicated("id").sum(), "PAD > PAS": (datos["pad_mmHg"] > datos["pas_mmHg"]).sum()}
ejes[1].barh(list(problemas), list(problemas.values()), color="firebrick")
ejes[1].set_title("Otros problemas del registro", fontweight="bold")
plt.tight_layout()
plt.show()

#### Desarrollo para el estudio

Tres centros de salud han reunido un **registro de riesgo cardiovascular** con 400 pacientes seguidos durante cinco años (datos sintéticos). Antes de analizarlo, una enfermera de investigación calcula el peso medio y obtiene 78,1 kg. Parece razonable.

Pero al revisar los datos aparecen un paciente de **680 kg**, otro de **7,4 kg**, cuatro tallas de «1,6 cm», una presión diastólica mayor que la sistólica, una glucemia de **1.120 mg/dL**, el sexo escrito como «Mujer» y como «mujer», 18 pacientes sin peso y dos pacientes registrados dos veces.

Ninguno de estos problemas impide calcular una media: Python la calcula sin protestar. Por eso son peligrosos. Se estima que la preparación de los datos ocupa la mayor parte del tiempo de cualquier análisis, y es la parte que más condiciona sus conclusiones.

## <font color='steelblue'>2. Conceptos clave (10 min)</font>

### <font color='steelblue'>2.1 Pandas y el DataFrame</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">Anatomía de un DataFrame</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_100_dataframe.png" width="780" alt="Anatomía de un DataFrame"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Filas = pacientes; columnas = variables; cada columna es una <b>Series</b> con su tipo.</li><li>NaN marca un dato ausente: pandas lo excluye de las medias y lo cuenta con .isna().</li></ul></div>

#### Desarrollo para el estudio

**Pandas** es la biblioteca de Python para trabajar con datos en forma de tabla. Su estructura central es el **DataFrame**: una tabla en la que cada **fila** es una unidad de observación (un paciente) y cada **columna** es una variable. Cada columna individual es una **Series**.

| Instrucción | Qué hace |
| --- | --- |
| `pd.read_csv(ruta)` | Carga un CSV en un DataFrame |
| `df.head()` | Muestra las primeras filas |
| `df.shape` | Número de filas y columnas |
| `df.dtypes` / `df.info()` | Tipo de cada columna y valores no nulos |
| `df.describe()` | Resumen numérico de cada columna |
| `df["col"]` | Selecciona una columna |
| `df[df["edad"] >= 65]` | Filtra filas que cumplen una condición |
| `df["col"].value_counts()` | Frecuencias de cada valor |

### <font color='steelblue'>2.2 Valores perdidos</font>

Pandas representa los datos ausentes como `NaN` (*not a number*). Antes de decidir qué hacer con ellos hay que preguntarse **por qué faltan**:

| Mecanismo | Significado | Ejemplo clínico | Consecuencia |
| --- | --- | --- | --- |
| Completamente al azar | La ausencia no depende de nada | Se perdió un tubo de la muestra | Eliminar los casos no sesga, solo reduce la muestra |
| Al azar condicionado | Depende de otras variables registradas | A los más jóvenes se les pide menos colesterol | Hay que tener en cuenta esas variables |
| No al azar | Depende del propio valor ausente | Los pacientes más obesos evitan pesarse | Cualquier tratamiento simple sesga el resultado |

**Rellenar con cero nunca es correcto** para una variable clínica. Rellenar con la media (imputación simple) mantiene el tamaño de la muestra, pero reduce artificialmente la variabilidad.

### <font color='steelblue'>2.3 Errores de registro y valores atípicos</font>

<div style="border-left:8px solid #4682B4;background:#F3F7FB;padding:18px 26px;border-radius:8px"><p style="font-size:30px;font-weight:700;color:#1F3B57;margin:0 0 12px 0">¿Error de registro o valor atípico real?</p><img src="https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/figuras/tema_100_plausible.png" width="780" alt="¿Error de registro o valor atípico real?"><ul style="font-size:21px;line-height:1.55;margin:14px 0 0 0"><li>Un error es imposible o incoherente: se corrige si se puede justificar o se convierte en NaN.</li><li>Un atípico real es extremo pero posible: se conserva. Eliminarlo sesga el análisis.</li></ul></div>

#### Desarrollo para el estudio

| | Error de registro | Valor atípico real |
| --- | --- | --- |
| Qué es | Un valor **imposible** o incompatible | Un valor **extremo pero posible** |
| Ejemplo | Peso de 680 kg, PAD mayor que PAS | Colesterol total de 340 mg/dL |
| Cómo se detecta | Rangos de plausibilidad fisiológica y reglas de coherencia | Criterios estadísticos, como el rango intercuartílico |
| Qué hacer | Corregir si la causa es evidente; si no, marcar como perdido | **Conservar**: es información clínica |

El **rango intercuartílico** (RIC) es la distancia entre el percentil 25 (Q1) y el 75 (Q3). El criterio de Tukey considera atípicos los valores por debajo de Q1 − 1,5 × RIC o por encima de Q3 + 1,5 × RIC. Lo estudiaremos con detalle en el Tema 120.

**Toda decisión de limpieza se documenta.** Un análisis que no explica qué datos cambió no es reproducible.

## <font color='steelblue'>3. Demostración: limpieza de un registro clínico (20 min)</font>

### <font color='steelblue'>3.1 Cargar y explorar</font>

In [ ]:
import pandas as pd
import numpy as np

URL = "https://raw.githubusercontent.com/fborrasumh/bioestadistica/main/datos/cohorte_cardiovascular.csv"
cohorte = pd.read_csv(URL)

print("Filas y columnas:", cohorte.shape)
cohorte.head()

In [ ]:
cohorte.info()

Observa en `info()` que algunas columnas tienen menos valores no nulos que filas: son los **valores perdidos**.

In [ ]:
cohorte.describe().round(1)

`describe()` ya delata los problemas: un peso máximo de 680 kg, una talla mínima de 1,6 cm, una PAD máxima de 163 mmHg y una glucemia máxima de 1.120 mg/dL.

### <font color='steelblue'>3.2 Seleccionar, filtrar y crear variables</font>

In [ ]:
# Una columna
print(cohorte["edad"].head(3))

# Varias columnas
print(cohorte[["id", "edad", "pas_mmHg"]].head(3))

# Filtrar: pacientes de 75 años o más del CS Centro
mayores = cohorte[(cohorte["edad"] >= 75) & (cohorte["centro"] == "CS Centro")]
print("Pacientes de 75 años o más en CS Centro:", len(mayores))

Al combinar condiciones en Pandas se usan `&` (y), `|` (o) y `~` (no), con **cada condición entre paréntesis**.

### <font color='steelblue'>3.3 Registro de limpieza</font>

Antes de cambiar nada, creamos una lista donde anotar cada decisión.

In [ ]:
registro_limpieza = []

def anotar(problema, decision, filas_afectadas):
    registro_limpieza.append({"problema": problema, "decisión": decision, "filas": filas_afectadas})

### <font color='steelblue'>3.4 Duplicados</font>

In [ ]:
duplicados = cohorte.duplicated().sum()
print("Filas duplicadas:", duplicados)

cohorte = cohorte.drop_duplicates()
anotar("Filas duplicadas", "Eliminadas", int(duplicados))
print("Pacientes únicos:", len(cohorte), "·", cohorte["id"].is_unique)

### <font color='steelblue'>3.5 Categorías escritas de formas distintas</font>

In [ ]:
print(cohorte["sexo"].value_counts())
print(cohorte["fumador"].value_counts(dropna=False))

In [ ]:
cambiados = (~cohorte["sexo"].isin(["Mujer", "Hombre"])).sum()
cohorte["sexo"] = cohorte["sexo"].str.strip().str.lower().map(
    {"mujer": "Mujer", "m": "Mujer", "hombre": "Hombre", "h": "Hombre"}
)
anotar("Sexo con distintas grafías", "Unificado a Mujer/Hombre", int(cambiados))

cambiados = (cohorte["fumador"].notna() & ~cohorte["fumador"].isin(["Sí", "No"])).sum()
cohorte["fumador"] = cohorte["fumador"].str.lower().map({"sí": "Sí", "si": "Sí", "no": "No"})
anotar("Fumador con distintas grafías", "Unificado a Sí/No", int(cambiados))

print(cohorte["sexo"].value_counts(), cohorte["fumador"].value_counts(dropna=False), sep="\n\n")

`.map()` sustituye cada valor según un diccionario. Un valor que no esté en el diccionario se convertiría en `NaN`; por eso conviene comprobar con `value_counts()` antes y después.

### <font color='steelblue'>3.6 Valores perdidos</font>

In [ ]:
perdidos = pd.DataFrame({
    "n_perdidos": cohorte.isna().sum(),
    "porcentaje": (cohorte.isna().mean() * 100).round(1),
})
perdidos[perdidos["n_perdidos"] > 0]

Veamos cómo cambia el peso medio según la decisión que tomemos. Primero corregiremos los errores de peso (apartado 3.7); aquí solo mostramos el efecto de **rellenar con cero**:

In [ ]:
peso = cohorte["peso_kg"]
print(f"Media ignorando los perdidos: {peso.mean():.1f} kg (n = {peso.notna().sum()})")
print(f"Media rellenando con cero:    {peso.fillna(0).mean():.1f} kg (n = {len(peso)})  ← sesgada")

Pandas **ignora los `NaN`** por defecto al calcular medias y otros estadísticos. Es lo correcto si los datos faltan al azar; en otro caso, hay que estudiar el patrón de ausencias. Lo que nunca se hace es sustituirlos por cero.

### <font color='steelblue'>3.7 Errores de registro: reglas de plausibilidad</font>

Definimos rangos fisiológicamente plausibles para un adulto y revisamos los valores que quedan fuera.

In [ ]:
rangos = {
    "peso_kg": (30, 250),
    "talla_cm": (120, 220),
    "pas_mmHg": (70, 260),
    "pad_mmHg": (30, 150),
    "glucemia_mg_dl": (40, 600),
}

for variable, (minimo, maximo) in rangos.items():
    fuera = cohorte[(cohorte[variable] < minimo) | (cohorte[variable] > maximo)]
    print(f"{variable}: {len(fuera)} valores fuera de [{minimo}, {maximo}]")
    if len(fuera) > 0:
        print("   ", fuera[variable].tolist())

Cada error requiere una decisión razonada:

- **Tallas entre 1 y 2:** se registraron en metros. La corrección es evidente: multiplicar por 100.
- **Pesos de 680 y 7,4 kg:** probablemente una coma desplazada (68,0 y 74), pero no podemos estar seguros. Los marcamos como perdidos.
- **Glucemia de 1.120 mg/dL:** probablemente un cero de más, pero sin certeza. La marcamos como perdida.

In [ ]:
en_metros = cohorte["talla_cm"] < 3
cohorte.loc[en_metros, "talla_cm"] = cohorte.loc[en_metros, "talla_cm"] * 100
anotar("Talla registrada en metros", "Multiplicada por 100", int(en_metros.sum()))

imposible = (cohorte["peso_kg"] < 30) | (cohorte["peso_kg"] > 250)
cohorte.loc[imposible, "peso_kg"] = np.nan
anotar("Peso fisiológicamente imposible", "Marcado como perdido", int(imposible.sum()))

imposible = cohorte["glucemia_mg_dl"] > 600
cohorte.loc[imposible, "glucemia_mg_dl"] = np.nan
anotar("Glucemia > 600 mg/dL", "Marcada como perdida", int(imposible.sum()))

**Regla de coherencia:** la presión diastólica no puede ser mayor que la sistólica. Si lo es, lo más probable es que se hayan intercambiado.

In [ ]:
invertidas = cohorte["pad_mmHg"] >= cohorte["pas_mmHg"]
print(cohorte.loc[invertidas, ["id", "pas_mmHg", "pad_mmHg"]])

cohorte.loc[invertidas, ["pas_mmHg", "pad_mmHg"]] = cohorte.loc[invertidas, ["pad_mmHg", "pas_mmHg"]].values
anotar("PAD mayor que PAS", "Valores intercambiados", int(invertidas.sum()))

`.loc[filas, columnas]` selecciona y modifica a la vez filas y columnas concretas. Es la forma segura de cambiar valores en un DataFrame.

### <font color='steelblue'>3.8 Valores atípicos reales: el criterio del RIC</font>

In [ ]:
q1, q3 = cohorte["colesterol_mg_dl"].quantile([0.25, 0.75])
ric = q3 - q1
limite_inferior, limite_superior = q1 - 1.5 * ric, q3 + 1.5 * ric

atipicos = cohorte[(cohorte["colesterol_mg_dl"] < limite_inferior) | (cohorte["colesterol_mg_dl"] > limite_superior)]
print(f"Límites de Tukey: {limite_inferior:.0f} a {limite_superior:.0f} mg/dL")
print("Valores atípicos:", sorted(atipicos["colesterol_mg_dl"].tolist()))

Un colesterol total de 84 mg/dL es inusual, pero **posible**. Es un valor atípico estadístico, no un error, y se **conserva**: eliminarlos ocultaría precisamente a los pacientes de mayor interés clínico.

### <font color='steelblue'>3.9 Variables derivadas y resultado final</font>

In [ ]:
cohorte["imc"] = (cohorte["peso_kg"] / (cohorte["talla_cm"] / 100) ** 2).round(1)
cohorte.describe().round(1)

In [ ]:
pd.DataFrame(registro_limpieza)

In [ ]:
cohorte.to_csv("cohorte_limpia.csv", index=False)
print("Guardado cohorte_limpia.csv con", len(cohorte), "pacientes")

### <font color='steelblue'>Ampliación: auditar una limpieza hecha por IA</font>

Si pides al asistente «limpia este conjunto de datos», es probable que proponga eliminar todas las filas con valores perdidos (`dropna()`) o los atípicos según algún criterio. Antes de aceptarlo:

1. Pide que **explique cada paso** y la razón clínica o estadística.
2. Comprueba **cuántas filas** se pierden: `dropna()` sobre toda la tabla puede eliminar un tercio de los pacientes.
3. Comprueba que no se eliminan **valores atípicos reales**.
4. Compara cinco valores antes y después, a mano.
5. Exige un **registro de limpieza** como el del apartado 3.3.

In [ ]:
original = pd.read_csv(URL)
print("Pacientes originales:", len(original))
print("Pacientes tras dropna() sobre toda la tabla:", len(original.dropna()))

### <font color='steelblue'>Ampliación: imputación por la media</font>

In [ ]:
hdl = cohorte["hdl_mg_dl"]
hdl_imputado = hdl.fillna(hdl.mean())
print(f"Desviación típica original:  {hdl.std():.2f} mg/dL")
print(f"Desviación típica imputada:  {hdl_imputado.std():.2f} mg/dL  ← subestimada")

Rellenar con la media no cambia la media, pero **reduce la variabilidad**: los intervalos de confianza y las pruebas estadísticas parecerán más precisos de lo que son. Existen métodos mejores, como la imputación múltiple, que quedan fuera de este curso.

### <font color='steelblue'>Ampliación: errores frecuentes</font>

**Rellenar con cero** los valores perdidos de una variable clínica.

**Eliminar todos los valores atípicos** sin distinguir errores de valores extremos reales.

**Usar `and`/`or` en lugar de `&`/`|`** al filtrar con Pandas: produce `ValueError`.

**Olvidar los paréntesis** en las condiciones combinadas: `cohorte["edad"] >= 65 & cohorte["diabetes"] == "Sí"` no hace lo que parece.

**Modificar una copia en lugar del original:** para cambiar valores, usar siempre `.loc[filas, columna]`.

**No documentar** los cambios: el análisis deja de ser reproducible.

## <font color='steelblue'>4. Práctica en la app Bioestadística (10 min)</font>

Abre la **[unidad 100 de la app Bioestadística](https://fborrasumh.github.io/bioestadistica/)** y completa sus misiones:

1. **Misión básica (sin asistente):** clasificar valores como error de registro, valor atípico real o dato plausible.
2. **Misión clínica (con pistas):** limpiar un registro sintético de pacientes con EPOC y presentar su registro de limpieza.
3. **Misión reto (con asistente):** pedir al asistente que limpie el registro, auditar su propuesta con los cinco pasos de la ampliación y corregir lo que no sea aceptable.

Al terminar, exporta el **informe de la unidad**.

## <font color='steelblue'>5. Autoevaluación (5 min)</font>

Responde sin asistente y sin ejecutar código.

In [ ]:
#@title Pregunta interactiva
preguntas = [{'question': 'En un registro, un paciente pesa 680 kg. ¿Qué es lo más adecuado?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Marcarlo como perdido y anotarlo en el registro de limpieza',
               'correct': True,
               'feedback': 'Correcto: es imposible y no sabemos con certeza el valor real.'},
              {'answer': 'Conservarlo, porque es un valor atípico',
               'correct': False,
               'feedback': 'No es un valor extremo posible: es un error de registro.'},
              {'answer': 'Sustituirlo por 68 kg',
               'correct': False,
               'feedback': 'Es probable, pero no seguro; corregir sin certeza introduce un dato '
                           'inventado.'},
              {'answer': 'Eliminar al paciente del estudio',
               'correct': False,
               'feedback': 'Se perderían todos sus demás datos, que son válidos.'}]},
 {'question': 'Un paciente tiene un colesterol total de 340 mg/dL, por encima del límite de Tukey. '
              '¿Qué haces?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Conservarlo: es un valor extremo pero clínicamente posible',
               'correct': True,
               'feedback': 'Correcto: es un valor atípico real con interés clínico.'},
              {'answer': 'Eliminarlo porque es un valor atípico',
               'correct': False,
               'feedback': 'Eliminarlo ocultaría a los pacientes de mayor riesgo.'},
              {'answer': 'Sustituirlo por la mediana',
               'correct': False,
               'feedback': 'Se perdería información real.'},
              {'answer': 'Marcarlo como perdido',
               'correct': False,
               'feedback': 'No es un error de registro.'}]},
 {'question': '¿Qué expresión selecciona a los pacientes de 65 años o más que son fumadores?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'cohorte[(cohorte["edad"] >= 65) & (cohorte["fumador"] == "Sí")]',
               'correct': True,
               'feedback': 'Correcto: & y cada condición entre paréntesis.'},
              {'answer': 'cohorte[cohorte["edad"] >= 65 and cohorte["fumador"] == "Sí"]',
               'correct': False,
               'feedback': 'En Pandas se usa & en lugar de and.'},
              {'answer': 'cohorte[(cohorte["edad"] >= 65) | (cohorte["fumador"] == "Sí")]',
               'correct': False,
               'feedback': '| selecciona a quienes cumplen al menos una condición.'},
              {'answer': 'cohorte["edad" >= 65 & "fumador" == "Sí"]',
               'correct': False,
               'feedback': 'Hay que comparar las columnas, no sus nombres.'}]},
 {'question': 'Faltan 18 pesos de 400. ¿Qué efecto tiene rellenarlos con 0 antes de calcular el '
              'peso medio?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Infraestima el peso medio',
               'correct': True,
               'feedback': 'Correcto: se introducen 18 pesos falsos de 0 kg.'},
              {'answer': 'Ninguno, porque Pandas ignora los ceros',
               'correct': False,
               'feedback': 'Pandas ignora los NaN, no los ceros.'},
              {'answer': 'Sobreestima el peso medio',
               'correct': False,
               'feedback': 'Los ceros arrastran la media hacia abajo.'},
              {'answer': 'Aumenta la precisión del resultado',
               'correct': False,
               'feedback': 'Introduce un sesgo.'}]},
 {'question': '¿Por qué no conviene eliminar todas las filas con algún valor perdido (dropna sobre '
              'toda la tabla)?',
  'type': 'multiple_choice',
  'answers': [{'answer': 'Porque se pierden muchos pacientes con datos válidos en otras variables',
               'correct': True,
               'feedback': 'Correcto: en el ejemplo se pierde casi una cuarta parte de la '
                           'cohorte.'},
              {'answer': 'Porque dropna produce un error con datos clínicos',
               'correct': False,
               'feedback': 'dropna funciona; el problema es lo que elimina.'},
              {'answer': 'Porque dropna convierte los NaN en ceros',
               'correct': False,
               'feedback': 'dropna elimina filas, no las rellena.'},
              {'answer': 'Porque Pandas no permite eliminar filas',
               'correct': False,
               'feedback': 'Sí lo permite.'}]}]
display_quiz(preguntas)

## <font color='steelblue'>Ideas clave</font>

- Un DataFrame es una tabla: una fila por paciente, una columna por variable.
- `info()`, `describe()` y `value_counts()` son la primera revisión obligatoria de cualquier conjunto de datos.
- Los valores perdidos se cuantifican y se tratan según por qué faltan; nunca se rellenan con cero.
- Un error de registro se corrige o se marca como perdido; un valor atípico real se conserva.
- Cada decisión de limpieza se anota en un registro.

## <font color='steelblue'>Glosario</font>

| Término | Definición |
| --- | --- |
| DataFrame | Tabla de datos de Pandas, con filas y columnas |
| Series | Columna de un DataFrame |
| `NaN` | Valor perdido en Pandas |
| Imputación | Sustitución de un valor perdido por un valor estimado |
| Error de registro | Valor imposible o incoherente, debido a un fallo al registrar el dato |
| Valor atípico | Valor extremo, alejado del resto, pero posible |
| Rango intercuartílico | Diferencia entre los percentiles 75 y 25 |
| Registro de limpieza | Documento que recoge cada cambio realizado en los datos y su motivo |

## <font color='steelblue'>Para saber más</font>

- [Guía de inicio de Pandas](https://pandas.pydata.org/docs/getting_started/index.html)
- [Pandas: trabajar con datos perdidos](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- Sterne JAC, *et al.* Multiple imputation for missing data in epidemiological and clinical research: potential and pitfalls. *BMJ*. 2009;338:b2393.

## <font color='steelblue'>Retos de este tema en la app</font>

Practica lo que has visto con las tres misiones del Tema 100 en la app **Bioestadística**. Cuentan para la evaluación continua.

1. **Básica (sin ayuda):** ¿Error de registro, atípico real o dato plausible?
2. **Clínica (con pistas):** Limpiar un registro de pacientes con EPOC
3. **Reto (con asistente de IA):** Auditar la limpieza que propone el asistente

<a href="https://fborrasumh.github.io/bioestadistica/#tema-100" target="_blank" style="display:inline-block;padding:10px 18px;background:#C8461F;color:#fff;border-radius:8px;font-weight:bold;text-decoration:none">Abrir los retos del Tema 100 en la app ↗</a>

---
*Tema 100 · Bioestadística (1964) · Grado en Medicina · Universidad Miguel Hernández de Elche · Fernando Borrás Rocher · CC BY 4.0*